# 02 Quality Analysis

This notebook validates the cleaned job-market dataset before SQL modeling and Power BI dashboard work.

## Goals

- Compare raw rows vs. cleaned rows.
- Verify duplicate removal and relationship integrity.
- Measure completeness for salary, skills, location, seniority, and remote status.
- Create BI-ready quality outputs and recommendations.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

PROJECT_ROOT = Path('..').resolve()
RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'job_postings_kaggle.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
JOBS_PATH = PROCESSED_DIR / 'job_postings_clean.csv'
SKILLS_PATH = PROCESSED_DIR / 'skills_clean.csv'
JOB_SKILLS_PATH = PROCESSED_DIR / 'job_skills_clean.csv'
QUALITY_DIR = PROCESSED_DIR / 'quality'
DOCS_DIR = PROJECT_ROOT / 'docs'

for path in [RAW_DATA_PATH, JOBS_PATH, SKILLS_PATH, JOB_SKILLS_PATH]:
    print(path.resolve(), 'exists:', path.exists())

C:\.projects\job-market-bi\data\raw\job_postings_kaggle.csv exists: True
C:\.projects\job-market-bi\data\processed\job_postings_clean.csv exists: True
C:\.projects\job-market-bi\data\processed\skills_clean.csv exists: True
C:\.projects\job-market-bi\data\processed\job_skills_clean.csv exists: True


## Load Data

Load only the fields needed for quality analysis so the notebook stays lightweight enough to rerun.

In [2]:
raw_key_columns = ['job_id', 'title', 'company_name', 'location', 'date_time']
job_quality_columns = [
    'job_id', 'title_clean', 'company_name_clean', 'location_clean',
    'posting_source_clean', 'schedule_type_simplified',
    'remote_status', 'is_remote', 'is_broad_location',
    'location_city', 'location_state', 'location_country',
    'estimated_posted_date', 'posting_age_days',
    'salary_period', 'salary_midpoint_annual', 'salary_band', 'has_salary_info',
    'skills_clean', 'skills_count', 'has_skills',
    'job_title_category', 'seniority_level', 'experience_years_min',
    'has_skill_sql', 'has_skill_python', 'has_skill_excel', 'has_skill_tableau', 'has_skill_power_bi'
]

raw_df = pd.read_csv(RAW_DATA_PATH, usecols=raw_key_columns)
jobs_df = pd.read_csv(JOBS_PATH, usecols=lambda col: col in job_quality_columns, low_memory=False)
skills_df = pd.read_csv(SKILLS_PATH)
job_skills_df = pd.read_csv(JOB_SKILLS_PATH, usecols=['job_id', 'skill_name', 'skill_category'])

for frame in [raw_df, jobs_df, job_skills_df]:
    frame['job_id'] = frame['job_id'].astype(str)

jobs_df['skills_count'] = pd.to_numeric(jobs_df['skills_count'], errors='coerce').fillna(0).astype(int)
jobs_df['salary_midpoint_annual'] = pd.to_numeric(jobs_df['salary_midpoint_annual'], errors='coerce')
jobs_df['posting_age_days'] = pd.to_numeric(jobs_df['posting_age_days'], errors='coerce')

print('Raw rows:', len(raw_df))
print('Cleaned job rows:', len(jobs_df))
print('Skills dimension rows:', len(skills_df))
print('Job-skill bridge rows:', len(job_skills_df))

Raw rows: 61953
Cleaned job rows: 58701
Skills dimension rows: 134
Job-skill bridge rows: 189085


## Quality Scorecard

High-level metrics that should be reviewed before dashboarding.

In [3]:
def pct(numerator, denominator):
    return np.nan if denominator == 0 else numerator / denominator

raw_rows = len(raw_df)
clean_rows = len(jobs_df)
raw_job_id_duplicates = raw_df.duplicated(subset=['job_id']).sum()
clean_job_id_duplicates = jobs_df.duplicated(subset=['job_id']).sum()

scorecard_rows = [
    {'metric': 'Raw rows', 'value': raw_rows, 'percentage': np.nan, 'notes': 'Rows in original Kaggle CSV'},
    {'metric': 'Cleaned rows', 'value': clean_rows, 'percentage': np.nan, 'notes': 'Rows after cleaning and deduplication'},
    {'metric': 'Rows removed', 'value': raw_rows - clean_rows, 'percentage': pct(raw_rows - clean_rows, raw_rows), 'notes': 'Removed during deduplication'},
    {'metric': 'Raw duplicate job_id rows', 'value': raw_job_id_duplicates, 'percentage': pct(raw_job_id_duplicates, raw_rows), 'notes': 'Duplicate job_id rows before cleaning'},
    {'metric': 'Clean duplicate job_id rows', 'value': clean_job_id_duplicates, 'percentage': pct(clean_job_id_duplicates, clean_rows), 'notes': 'Should be zero'},
    {'metric': 'Unique companies', 'value': jobs_df['company_name_clean'].nunique(dropna=True), 'percentage': np.nan, 'notes': 'Distinct cleaned company names'},
    {'metric': 'Unique locations', 'value': jobs_df['location_clean'].nunique(dropna=True), 'percentage': np.nan, 'notes': 'Distinct cleaned location strings'},
    {'metric': 'Unique skills', 'value': len(skills_df), 'percentage': np.nan, 'notes': 'Rows in skills dimension'},
    {'metric': 'Job-skill assignments', 'value': len(job_skills_df), 'percentage': np.nan, 'notes': 'Rows in bridge table'},
    {'metric': 'Jobs with at least one skill', 'value': jobs_df['has_skills'].sum(), 'percentage': jobs_df['has_skills'].mean(), 'notes': 'Skill extraction coverage'},
    {'metric': 'Jobs with salary', 'value': jobs_df['has_salary_info'].sum(), 'percentage': jobs_df['has_salary_info'].mean(), 'notes': 'Salary-listed postings only'},
    {'metric': 'Jobs with unknown seniority', 'value': (jobs_df['seniority_level'] == 'Unknown').sum(), 'percentage': (jobs_df['seniority_level'] == 'Unknown').mean(), 'notes': 'Unknown should remain visible in dashboard'},
    {'metric': 'Broad/unclear locations', 'value': jobs_df['is_broad_location'].sum(), 'percentage': jobs_df['is_broad_location'].mean(), 'notes': 'Anywhere or United States'},
    {'metric': 'Remote jobs', 'value': jobs_df['is_remote'].sum(), 'percentage': jobs_df['is_remote'].mean(), 'notes': 'Explicit remote plus Anywhere-inferred remote'},
    {'metric': 'Jobs with estimated posted date', 'value': jobs_df['estimated_posted_date'].notna().sum(), 'percentage': jobs_df['estimated_posted_date'].notna().mean(), 'notes': 'Estimated from date_time and posted_at'},
]

quality_scorecard = pd.DataFrame(scorecard_rows)
quality_scorecard['percentage_display'] = quality_scorecard['percentage'].map(lambda value: '' if pd.isna(value) else f'{value:.1%}')
quality_scorecard

,metric,value,percentage,notes,percentage_display
0,Raw rows,61953,NaN,Rows in original Kaggle CSV,
1,Cleaned rows,58701,NaN,Rows after cleaning and deduplication,
2,Rows removed,3252,0.052491,Removed during deduplication,5.2%
3,Raw duplicate job_id rows,3178,0.051297,Duplicate job_id rows before cleaning,5.1%
4,Clean duplicate job_id rows,0,0.000000,Should be zero,0.0%
5,Unique companies,13428,NaN,Distinct cleaned company names,
6,Unique locations,970,NaN,Distinct cleaned location strings,
7,Unique skills,134,NaN,Rows in skills dimension,
8,Job-skill assignments,189085,NaN,Rows in bridge table,
9,Jobs with at least one skill,45915,0.782184,Skill extraction coverage,78.2%


## Completeness Report

Measure which fields are ready for dashboard filters, relationships, and measures.

In [4]:
completeness_columns = [
    'job_id', 'title_clean', 'company_name_clean', 'location_clean',
    'posting_source_clean', 'schedule_type_simplified', 'remote_status',
    'location_country', 'estimated_posted_date', 'salary_midpoint_annual',
    'skills_clean', 'job_title_category', 'seniority_level'
]

completeness_report = pd.DataFrame({
    'column': completeness_columns,
    'non_null_rows': [jobs_df[col].notna().sum() for col in completeness_columns],
    'missing_rows': [jobs_df[col].isna().sum() for col in completeness_columns],
})
completeness_report['completeness_pct'] = completeness_report['non_null_rows'] / len(jobs_df)
completeness_report = completeness_report.sort_values('completeness_pct')
completeness_report

,column,non_null_rows,missing_rows,completeness_pct
9,salary_midpoint_annual,9660,49041,0.164563
10,skills_clean,45915,12786,0.782184
7,location_country,58272,429,0.992692
8,estimated_posted_date,58650,51,0.999131
3,location_clean,58664,37,0.999370
4,posting_source_clean,58692,9,0.999847
0,job_id,58701,0,1.000000
1,title_clean,58701,0,1.000000
2,company_name_clean,58701,0,1.000000
5,schedule_type_simplified,58701,0,1.000000


## Integrity Checks

These checks protect SQL and Power BI modeling from broken relationships or misleading fields.

In [5]:
job_ids = set(jobs_df['job_id'])
bridge_job_ids = set(job_skills_df['job_id'])
skill_names = set(skills_df['skill_name'])
bridge_skill_names = set(job_skills_df['skill_name'])

bridge_counts = job_skills_df.groupby('job_id').size().rename('bridge_skill_count')
skill_count_check = jobs_df[['job_id', 'skills_count']].merge(bridge_counts, how='left', left_on='job_id', right_index=True)
skill_count_check['bridge_skill_count'] = skill_count_check['bridge_skill_count'].fillna(0).astype(int)
skill_count_mismatches = (skill_count_check['skills_count'] != skill_count_check['bridge_skill_count']).sum()

salary_without_midpoint = jobs_df['has_salary_info'] & jobs_df['salary_midpoint_annual'].isna()
no_salary_with_midpoint = (~jobs_df['has_salary_info']) & jobs_df['salary_midpoint_annual'].notna()

def check_status(issue_count, warning_threshold=0):
    return 'PASS' if issue_count <= warning_threshold else 'FAIL'

quality_checks = pd.DataFrame([
    {'check': 'Cleaned row count is lower than raw row count', 'issue_count': 0 if clean_rows < raw_rows else 1, 'status': 'PASS' if clean_rows < raw_rows else 'WARN', 'notes': 'Cleaning should remove duplicate rows'},
    {'check': 'No duplicate job_id values in cleaned jobs', 'issue_count': int(clean_job_id_duplicates), 'status': check_status(clean_job_id_duplicates), 'notes': 'job_id should be unique'},
    {'check': 'All bridge job_ids exist in jobs table', 'issue_count': len(bridge_job_ids - job_ids), 'status': check_status(len(bridge_job_ids - job_ids)), 'notes': 'Protects jobs to job_skills relationship'},
    {'check': 'All bridge skill names exist in skills dimension', 'issue_count': len(bridge_skill_names - skill_names), 'status': check_status(len(bridge_skill_names - skill_names)), 'notes': 'Protects skills to job_skills relationship'},
    {'check': 'Job skills_count matches bridge rows', 'issue_count': int(skill_count_mismatches), 'status': check_status(skill_count_mismatches), 'notes': 'Flat job field should agree with bridge table'},
    {'check': 'Salary flag matches annual midpoint', 'issue_count': int(salary_without_midpoint.sum() + no_salary_with_midpoint.sum()), 'status': check_status(salary_without_midpoint.sum() + no_salary_with_midpoint.sum()), 'notes': 'has_salary_info should reflect salary_midpoint_annual'},
    {'check': 'Remote status is populated', 'issue_count': int(jobs_df['remote_status'].isna().sum()), 'status': check_status(jobs_df['remote_status'].isna().sum()), 'notes': 'Needed for remote/onsite reporting'},
    {'check': 'Job title category is populated', 'issue_count': int(jobs_df['job_title_category'].isna().sum()), 'status': check_status(jobs_df['job_title_category'].isna().sum()), 'notes': 'Needed for role analysis'},
])

quality_checks

,check,issue_count,status,notes
0,Cleaned row count is lower than raw row count,0,PASS,Cleaning should remove duplicate rows
1,No duplicate job_id values in cleaned jobs,0,PASS,job_id should be unique
2,All bridge job_ids exist in jobs table,0,PASS,Protects jobs to job_skills relationship
3,All bridge skill names exist in skills dimension,0,PASS,Protects skills to job_skills relationship
4,Job skills_count matches bridge rows,0,PASS,Flat job field should agree with bridge table
5,Salary flag matches annual midpoint,0,PASS,has_salary_info should reflect salary_midpoint...
6,Remote status is populated,0,PASS,Needed for remote/onsite reporting
7,Job title category is populated,0,PASS,Needed for role analysis


## BI Distribution Tables

These previews help validate whether categories are useful for dashboard slicing.

In [6]:
def distribution_table(df, column, top_n=20):
    result = df[column].value_counts(dropna=False).head(top_n).reset_index()
    result.columns = [column, 'job_count']
    result['percentage'] = result['job_count'] / len(df)
    return result

title_category_distribution = distribution_table(jobs_df, 'job_title_category')
seniority_distribution = distribution_table(jobs_df, 'seniority_level')
remote_distribution = distribution_table(jobs_df, 'remote_status')
salary_band_distribution = distribution_table(jobs_df, 'salary_band')
source_distribution = distribution_table(jobs_df, 'posting_source_clean')
location_distribution = distribution_table(jobs_df, 'location_clean')

top_skills = job_skills_df['skill_name'].value_counts().head(25).reset_index()
top_skills.columns = ['skill_name', 'job_count']
top_skills['percentage_of_jobs'] = top_skills['job_count'] / len(jobs_df)

skill_category_distribution = job_skills_df['skill_category'].value_counts().reset_index()
skill_category_distribution.columns = ['skill_category', 'assignment_count']
skill_category_distribution['percentage_of_assignments'] = skill_category_distribution['assignment_count'] / len(job_skills_df)

display(title_category_distribution)
display(seniority_distribution)
display(remote_distribution)
display(salary_band_distribution)
display(top_skills.head(15))

,job_title_category,job_count,percentage
0,Data Analyst,41117,0.700448
1,Other,7430,0.126574
2,Data Scientist,2881,0.049079
3,BI Analyst,2177,0.037086
4,Analytics Manager/Leader,1865,0.031771
5,Data Engineer,1329,0.022640
6,Business Analyst,1172,0.019966
7,BI Developer/Engineer,236,0.004020
8,Business Intelligence,183,0.003117
9,Operations Analyst,132,0.002249


,seniority_level,job_count,percentage
0,Unknown,31903,0.543483
1,Senior,13097,0.223114
2,Mid-Level,6080,0.103576
3,Entry-Level,5643,0.096131
4,Manager/Executive,1978,0.033696


,remote_status,job_count,percentage
0,Remote,26328,0.448510
1,Location-Based/Unclear,17254,0.293930
2,Broad Location - Unclear,15080,0.256895
3,Unknown,37,0.000630
4,Likely Remote - Anywhere,2,0.000034


,salary_band,job_count,percentage
0,No Salary Listed,49041,0.835437
1,50K-75K,2212,0.037682
2,75K-100K,2168,0.036933
3,100K-125K,2018,0.034378
4,<50K,1563,0.026626
5,125K-150K,882,0.015025
6,150K+,817,0.013918


,skill_name,job_count,percentage_of_jobs
0,SQL,29223,0.497828
1,Excel,18683,0.318274
2,Python,17809,0.303385
3,Tableau,15748,0.268275
4,Power BI,15115,0.257491
5,R,10697,0.182229
6,SAS,4968,0.084632
7,PowerPoint,4217,0.071839
8,Word,4074,0.069403
9,Azure,3725,0.063457


## BI Readiness Recommendations

Translate quality findings into dashboard rules.

In [7]:
salary_availability = jobs_df['has_salary_info'].mean()
skills_availability = jobs_df['has_skills'].mean()
unknown_seniority_pct = (jobs_df['seniority_level'] == 'Unknown').mean()
broad_location_pct = jobs_df['is_broad_location'].mean()

recommendations = [
    {
        'area': 'Salary',
        'recommendation': 'Use salary visuals only for salary-listed jobs and label them clearly.',
        'reason': f'Only {salary_availability:.1%} of cleaned postings include usable salary information.'
    },
    {
        'area': 'Skills',
        'recommendation': 'Use job_skills_clean.csv for skill analysis instead of the comma-separated skills_clean field.',
        'reason': f'{skills_availability:.1%} of postings have at least one extracted skill and the bridge table supports many-to-many analysis.'
    },
    {
        'area': 'Seniority',
        'recommendation': 'Keep Unknown seniority visible in all seniority charts.',
        'reason': f'{unknown_seniority_pct:.1%} of postings do not provide enough information for a confident seniority label.'
    },
    {
        'area': 'Location and Remote',
        'recommendation': 'Separate Remote, Broad Location - Unclear, and Location-Based/Unclear in dashboard filters.',
        'reason': f'{broad_location_pct:.1%} of postings use broad location values such as Anywhere or United States.'
    },
    {
        'area': 'Modeling',
        'recommendation': 'Use fact_jobs, dim_skills, and bridge_job_skills as the first SQL model layer.',
        'reason': 'Integrity checks validate the job to skill relationships and no duplicate cleaned job_id values remain.'
    },
]

recommendations_df = pd.DataFrame(recommendations)
recommendations_df

,area,recommendation,reason
0,Salary,Use salary visuals only for salary-listed jobs...,Only 16.5% of cleaned postings include usable ...
1,Skills,Use job_skills_clean.csv for skill analysis in...,78.2% of postings have at least one extracted ...
2,Seniority,Keep Unknown seniority visible in all seniorit...,54.3% of postings do not provide enough inform...
3,Location and Remote,"Separate Remote, Broad Location - Unclear, and...",70.5% of postings use broad location values su...
4,Modeling,"Use fact_jobs, dim_skills, and bridge_job_skil...",Integrity checks validate the job to skill rel...


## Export Quality Artifacts

Save reusable quality outputs for documentation and future SQL/Power BI work.

In [8]:
QUALITY_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

quality_scorecard.to_csv(QUALITY_DIR / 'quality_scorecard.csv', index=False)
completeness_report.to_csv(QUALITY_DIR / 'completeness_report.csv', index=False)
quality_checks.to_csv(QUALITY_DIR / 'quality_checks.csv', index=False)
recommendations_df.to_csv(QUALITY_DIR / 'quality_recommendations.csv', index=False)
title_category_distribution.to_csv(QUALITY_DIR / 'job_title_category_distribution.csv', index=False)
seniority_distribution.to_csv(QUALITY_DIR / 'seniority_distribution.csv', index=False)
remote_distribution.to_csv(QUALITY_DIR / 'remote_status_distribution.csv', index=False)
salary_band_distribution.to_csv(QUALITY_DIR / 'salary_band_distribution.csv', index=False)
top_skills.to_csv(QUALITY_DIR / 'top_skills.csv', index=False)
skill_category_distribution.to_csv(QUALITY_DIR / 'skill_category_distribution.csv', index=False)

report_lines = [
    '# Data Quality Report',
    '',
    '## Executive Summary',
    f'- Raw rows: {raw_rows:,}',
    f'- Cleaned rows: {clean_rows:,}',
    f'- Rows removed: {raw_rows - clean_rows:,} ({pct(raw_rows - clean_rows, raw_rows):.1%})',
    f'- Jobs with salary: {int(jobs_df["has_salary_info"].sum()):,} ({salary_availability:.1%})',
    f'- Jobs with at least one skill: {int(jobs_df["has_skills"].sum()):,} ({skills_availability:.1%})',
    f'- Unique skills: {len(skills_df):,}',
    f'- Job-skill assignments: {len(job_skills_df):,}',
    '',
    '## BI Rules',
    '- Salary visuals must be labeled as salary-listed postings only.',
    '- Remote analysis must separate Remote from Broad Location - Unclear.',
    '- Seniority visuals must keep Unknown visible.',
    '- Skill visuals must use the job-skill bridge table.',
    '',
    '## Integrity Check Summary',
    f'- Checks passed: {(quality_checks["status"] == "PASS").sum()} of {len(quality_checks)}',
    f'- Checks needing review: {(quality_checks["status"] != "PASS").sum()} of {len(quality_checks)}',
    '',
    '## Output Files',
    '- data/processed/quality/quality_scorecard.csv',
    '- data/processed/quality/completeness_report.csv',
    '- data/processed/quality/quality_checks.csv',
    '- data/processed/quality/quality_recommendations.csv',
]

(DOCS_DIR / 'data_quality_report.md').write_text('\n'.join(report_lines), encoding='utf-8')

print('Saved quality outputs to', QUALITY_DIR.resolve())
print('Saved markdown report to', (DOCS_DIR / 'data_quality_report.md').resolve())

Saved quality outputs to C:\.projects\job-market-bi\data\processed\quality
Saved markdown report to C:\.projects\job-market-bi\docs\data_quality_report.md
